In [ ]:
! pip install pandas scikit-learn numpy
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("mainSimulationAccessTraces.csv")

print(data.shape)

(357952, 13)


In [3]:
print(data.columns)


Index(['sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
       'destinationServiceAddress', 'destinationServiceType',
       'destinationLocation', 'accessedNodeAddress', 'accessedNodeType',
       'operation', 'value', 'timestamp', 'normality'],
      dtype='object')


In [4]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 357952 entries, 0 to 357951
Data columns (total 13 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   sourceID                   357952 non-null  object
 1   sourceAddress              357952 non-null  object
 2   sourceType                 357952 non-null  object
 3   sourceLocation             357952 non-null  object
 4   destinationServiceAddress  357952 non-null  object
 5   destinationServiceType     357952 non-null  object
 6   destinationLocation        357952 non-null  object
 7   accessedNodeAddress        357952 non-null  object
 8   accessedNodeType           357804 non-null  object
 9   operation                  357952 non-null  object
 10  value                      355902 non-null  object
 11  timestamp                  357952 non-null  int64 
 12  normality                  357952 non-null  object
dtypes: int64(1), object(12)
memory usage: 35.5+ 

Nettoyage et simplification

In [5]:
drop_cols = [
    "sourceID",
    "sourceAddress",
    "destinationServiceAddress",
    "accessedNodeAddress"
]

data.drop(columns=drop_cols, inplace=True)


Vérification des valeurs manquantes

In [6]:
print(data.isna().sum())

sourceType                   0
sourceLocation               0
destinationServiceType       0
destinationLocation          0
accessedNodeType           148
operation                    0
value                     2050
timestamp                    0
normality                    0
dtype: int64


In [7]:
print(data['normality'].value_counts())

normality
normal                           347935
anomalous(DoSattack)               5780
anomalous(scan)                    1547
anomalous(malitiousControl)         889
anomalous(malitiousOperation)       805
anomalous(spying)                   532
anomalous(dataProbing)              342
anomalous(wrongSetUp)               122
Name: count, dtype: int64


In [8]:
#  0 = normal, 1 = anomalous
data['normality'] = data['normality'].apply(lambda x: 0 if x == 'normal' else 1)

# Check counts
print(data['normality'].value_counts())

normality
0    347935
1     10017
Name: count, dtype: int64


In [9]:
print(data['value'].value_counts())

value
1                    57047
0                    50282
false                25966
true                 14460
10                    9635
                     ...  
1.521217446778E12        1
1.521220297057E12        1
1.521220308496E12        1
1.521220325816E12        1
1.521225180883E12        1
Name: count, Length: 10623, dtype: int64


Détection Sémantique : probleme Value

In [10]:
np.set_printoptions(threshold=np.inf)
print(data['value'].unique())

['none' '20.0464' '1' '1.52121675521E12' '19.8391' '19.9088' '19.9523'
 '1.521216758314E12' '19.8185' '20.0005' '0' '20.4750' '20.2078' '20.1757'
 '19.7536' '19.7719' '20.4103' '19.8609' '20.1794' '20.0372' '20.1058'
 '19.7941' '19.9142' '20.3042' '20.1346' '19.7922' '20.1731' '20.1458'
 '19.7587' '20.1929' '20.4331' '19.6966' '19.6209' '19.9143' '19.6963'
 '19.5072' '20.4032' '1.521216770002E12' '19.9860' '20.2714' '20.3664'
 '19.6491' '19.6007' '19.7861' '20.2914' '19.8480' '20.3901' '19.5935'
 '19.9351' '20.4009' '19.5693' '19.9501' '20.3104' '19.6382' '20.2557'
 '19.7876' '20.0605' '19.9950' '20.0012' '20.0302' '19.5702' '20.3979'
 '19.6520' '19.8319' '20.4505' '19.6254' '20.0283' '19.6995' '20.1410'
 '20.4480' '20.1077' '1.521216782996E12' '19.8770' '19.9342' '20.4309'
 '19.8495' '20.4105' '20.1277' '19.7564' '19.8989' '19.5407' '20.1878'
 '19.5759' '19.8346' '20.4532' '19.6320' '19.8776' '19.6612' '19.9497'
 '20.2760' '19.6815' '20.0496' '20.1338' '20.2199' '20.2705' '20.2783'
 '

 Clean 'value' column

In [11]:
data['value'] = data['value'].astype(str).str.strip().str.lower()

data['value'] = data['value'].replace(
    ['none', 'null', 'nan', ''],
    np.nan
)


In [12]:
import re
JAVA_OBJ_REGEX = re.compile(r"^[a-zA-Z0-9_.]+@[a-f0-9]+$")

def detect_semantic_type(v):
    if v is None:
        return "missing"
    v = str(v).strip().lower()
    if v in ("none", "null", "nan", ""):
        return "missing"
    if v in ("true", "false"):
        return "boolean"
    if JAVA_OBJ_REGEX.match(v):
        return "java_object"

    # Try to parse as float to distinguish timestamp/float/integer
    try:
        f = float(v)
    except Exception:
        f = None

    if f is not None:
        # large numbers likely timestamp in ms
        if f > 1e11:
            return "timestamp_ms"
        # pure digit strings represent integers
        if v.isdigit():
            return "integer"
        # otherwise it's a float
        return "float"

    # fallback checks
    if v.isdigit():
        return "integer"
    return "string"

data["value_type"] = data["value"].apply(detect_semantic_type)
print(data["value_type"].value_counts())

value_type
integer         184469
float           130115
boolean          40426
missing           2156
timestamp_ms       575
string             200
java_object         11
Name: count, dtype: int64


Creation des features

In [13]:
#flags (type indicators)
for t in data['value_type'].unique():
    data[f'value_is_{t}'] = (data['value_type'] == t).astype(int)

In [14]:
data['value_int'] = np.nan
mask = data['value_type']=='integer'
data.loc[mask,'value_int'] = pd.to_numeric(data.loc[mask,'value'], errors='coerce')

data['value_float'] = np.nan
mask = data['value_type']=='float'
data.loc[mask,'value_float'] = pd.to_numeric(data.loc[mask,'value'], errors='coerce')

data['value_bool'] = np.nan
mask = data['value_type']=='boolean'
data.loc[mask,'value_bool'] = data.loc[mask,'value'].map({'true':1,'false':0})

# Timestamps
data['value_ts'] = pd.NaT
mask = data['value_type']=='timestamp_ms'
data.loc[mask,'value_ts'] = pd.to_datetime(pd.to_numeric(data.loc[mask,'value'], errors='coerce'), unit='ms', errors='coerce')
data['value_ts_hour'] = data['value_ts'].dt.hour
data['value_ts_minute'] = data['value_ts'].dt.minute

data['value_ts_hour'] = data['value_ts'].dt.hour
data['value_ts_minute'] = data['value_ts'].dt.minute

In [15]:
features_ext = [
    'value_int', 'value_float', 'value_bool',
    'value_ts', 'value_ts_hour', 'value_ts_minute'
]


type_flags = [col for col in data.columns if col.startswith('value_is_')]
print(type_flags)

display(data[features_ext + type_flags].head(10))

['value_is_missing', 'value_is_float', 'value_is_integer', 'value_is_timestamp_ms', 'value_is_boolean', 'value_is_java_object', 'value_is_string']


,value_int,value_float,value_bool,value_ts,value_ts_hour,value_ts_minute,value_is_missing,value_is_float,value_is_integer,value_is_timestamp_ms,value_is_boolean,value_is_java_object,value_is_string
0,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
1,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
2,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
3,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
4,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
5,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
6,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
7,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
8,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0
9,NaN,NaN,NaN,NaT,NaN,NaN,1,0,0,0,0,0,0


Extraction des features temporelles

In [16]:
data['datetime'] = pd.to_datetime(data['timestamp'], unit='ms', errors='coerce')
data['hour'] = data['datetime'].dt.hour
data['minute'] = data['datetime'].dt.minute
data['second'] = data['datetime'].dt.second
data['time_of_day'] = data['hour'] + data['minute']/60
data['is_night'] = data['hour'].apply(lambda h:1 if 22<=h or h<=6 else 0)

# Fill missing accessedNodeType
data['accessedNodeType'] = data['accessedNodeType'].fillna('unknown')

In [17]:
data['is_night'].value_counts()

is_night
0    238361
1    119591
Name: count, dtype: int64

In [18]:
time_features = ['datetime', 'hour', 'minute', 'second', 'time_of_day', 'is_night']

print(data[time_features].head(10))


                 datetime  hour  minute  second  time_of_day  is_night
0 2018-03-02 23:00:00.000    23       0       0         23.0         1
1 2018-03-02 23:00:03.269    23       0       3         23.0         1
2 2018-03-02 23:00:03.279    23       0       3         23.0         1
3 2018-03-02 23:00:03.290    23       0       3         23.0         1
4 2018-03-02 23:00:03.464    23       0       3         23.0         1
5 2018-03-02 23:00:03.464    23       0       3         23.0         1
6 2018-03-02 23:00:03.474    23       0       3         23.0         1
7 2018-03-02 23:00:03.474    23       0       3         23.0         1
8 2018-03-02 23:00:03.484    23       0       3         23.0         1
9 2018-03-02 23:00:04.212    23       0       4         23.0         1


In [19]:
# Afficher 10 lignes aléatoires
print(data[time_features].sample(10))

                      datetime  hour  minute  second  time_of_day  is_night
140377 2018-03-03 09:51:35.823     9      51      35     9.850000         0
218408 2018-03-03 14:41:08.512    14      41       8    14.683333         0
29609  2018-03-03 01:20:31.299     1      20      31     1.333333         1
47909  2018-03-03 02:41:23.680     2      41      23     2.683333         1
323153 2018-03-03 20:52:52.696    20      52      52    20.866667         0
158505 2018-03-03 11:07:04.002    11       7       4    11.116667         0
77509  2018-03-03 04:59:37.348     4      59      37     4.983333         1
236895 2018-03-03 15:49:19.684    15      49      19    15.816667         0
297797 2018-03-03 19:27:21.194    19      27      21    19.450000         0
210694 2018-03-03 14:12:36.757    14      12      36    14.200000         0


In [20]:
cols_to_show = [
    'datetime', 'hour', 'minute', 'second', 'time_of_day', 'is_night',
    'value_int', 'value_float', 'value_bool',
    'value_ts', 'value_ts_hour', 'value_ts_minute',
    'value_is_missing', 'value_is_float', 'value_is_integer', 'value_is_timestamp_ms', 'value_is_boolean', 'value_is_java_object', 'value_is_string',
    'sourceType', 'sourceLocation',
    'destinationServiceType', 'destinationLocation',
    'accessedNodeType', 'operation',
    'normality'
]

from IPython.display import display
display(data[cols_to_show].head(10))


,datetime,hour,minute,second,time_of_day,is_night,value_int,value_float,value_bool,value_ts,...,value_is_boolean,value_is_java_object,value_is_string,sourceType,sourceLocation,destinationServiceType,destinationLocation,accessedNodeType,operation,normality
0,2018-03-02 23:00:00.000,23,0,0,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/lightControler,BedroomParents,/lightControler,BedroomParents,/lightControler,registerService,0
1,2018-03-02 23:00:03.269,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/lightControler,Dinningroom,/lightControler,Dinningroom,/lightControler,registerService,0
2,2018-03-02 23:00:03.279,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/lightControler,BedroomChildren,/lightControler,BedroomChildren,/lightControler,registerService,0
3,2018-03-02 23:00:03.290,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/lightControler,Kitchen,/lightControler,Kitchen,/lightControler,registerService,0
4,2018-03-02 23:00:03.464,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/movementSensor,Kitchen,/movementSensor,Kitchen,/movementSensor,registerService,0
5,2018-03-02 23:00:03.464,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/sensorService,BedroomParents,/sensorService,BedroomParents,/sensorService,registerService,0
6,2018-03-02 23:00:03.474,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/movementSensor,BedroomParents,/movementSensor,BedroomParents,/movementSensor,registerService,0
7,2018-03-02 23:00:03.474,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/sensorService,Kitchen,/sensorService,Kitchen,/sensorService,registerService,0
8,2018-03-02 23:00:03.484,23,0,3,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/sensorService,BedroomChildren,/sensorService,BedroomChildren,/sensorService,registerService,0
9,2018-03-02 23:00:04.212,23,0,4,23.0,1,NaN,NaN,NaN,NaT,...,0,0,0,/movementSensor,BedroomChildren,/movementSensor,BedroomChildren,/movementSensor,registerService,0


In [21]:
# Count non-NaNs per column
print(data[['value_int','value_float','value_bool','value_ts']].notna().sum())

# Check type flags
data[[col for col in data.columns if col.startswith('value_is_')]].sum()


value_int      184469
value_float    130115
value_bool      40426
value_ts          575
dtype: int64


value_is_missing           2156
value_is_float           130115
value_is_integer         184469
value_is_timestamp_ms       575
value_is_boolean          40426
value_is_java_object         11
value_is_string             200
dtype: int64

Encodage des colonnes catégorielles

In [22]:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['sourceType','sourceLocation','destinationServiceType','destinationLocation','accessedNodeType','operation']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(data[categorical_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols))

# Tree-based dataset
data_tree = pd.concat([data.drop(columns=categorical_cols + ['value','timestamp','datetime','value_type']), encoded_df], axis=1)

In [23]:
display(data_tree.head(10))

,normality,value_is_missing,value_is_float,value_is_integer,value_is_timestamp_ms,value_is_boolean,value_is_java_object,value_is_string,value_int,value_float,...,accessedNodeType_/sensorService,accessedNodeType_/smartPhone,accessedNodeType_/thermostat,accessedNodeType_/washingService,accessedNodeType_unknown,operation_lockSubtree,operation_read,operation_registerService,operation_subscribe,operation_write
0,0,1,0,0,0,0,0,0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0,1,0,0,0,0,0,0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0,1,0,0,0,0,0,0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0,1,0,0,0,0,0,0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0,1,0,0,0,0,0,0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5,0,1,0,0,0,0,0,0,NaN,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6,0,1,0,0,0,0,0,0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7,0,1,0,0,0,0,0,0,NaN,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
8,0,1,0,0,0,0,0,0,NaN,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0,1,0,0,0,0,0,0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


Numeric imputation

In [24]:
from sklearn.impute import SimpleImputer
print(data[['value_int','value_float','value_bool','value_ts']].notna().sum())
numeric_features = ['value_int','value_float','value_bool','value_ts_hour','value_ts_minute','hour','minute','second','time_of_day','is_night']

# Tree-based: median
tree_imputer = SimpleImputer(strategy='median')
data_tree[numeric_features] = tree_imputer.fit_transform(data_tree[numeric_features])



# DL: fill missing numeric with 0
data_dl = data.drop(columns=['value','timestamp','datetime','value_type']).copy()
data_dl[numeric_features] = data_dl[numeric_features].fillna(0)

value_int      184469
value_float    130115
value_bool      40426
value_ts          575
dtype: int64


In [25]:
print(data_dl[['value_int','value_float','value_bool','value_ts']].isna().sum())

value_int           0
value_float         0
value_bool          0
value_ts       357377
dtype: int64


In [26]:
print(data_tree[['value_int','value_float','value_bool','value_ts']].isna().sum())

value_int           0
value_float         0
value_bool          0
value_ts       357377
dtype: int64


In [27]:
data_tree.drop(columns=[
    'value_ts','value_ts_hour','value_ts_minute'
], inplace=True)

data_dl.drop(columns=[
    'value_ts','value_ts_hour','value_ts_minute'
], inplace=True)

Categorical encoding for DL (embeddings)

In [28]:
from sklearn.preprocessing import LabelEncoder

data_dl_cat = data_dl[categorical_cols].copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data_dl_cat[col] = le.fit_transform(data_dl_cat[col])
    label_encoders[col] = le

# Drop original categorical columns and concat integer-encoded
data_dl = pd.concat([data_dl.drop(columns=categorical_cols), data_dl_cat], axis=1)

Normalize time features for DL

In [29]:
from sklearn.preprocessing import MinMaxScaler

normalize_cols = ['hour','minute','second','time_of_day']
scaler_minmax = MinMaxScaler()
data_dl[normalize_cols] = scaler_minmax.fit_transform(data_dl[normalize_cols])

Export datasets

In [30]:
data_tree.to_csv("data/data_processed_for_tree_models.csv", index=False)
data_dl.to_csv("data/data_processed_for_dl_models.csv", index=False)

print("Preprocessed datasets exported successfully!")


Preprocessed datasets exported successfully!


Résultat final

Features numériques / booléennes :
value_int, value_float, value_bool, value_is_missing, value_is_float, value_is_integer, 
value_is_timestamp_ms, value_is_boolean, value_is_java_object, value_is_string, 
value_ts_hour, value_ts_minute, hour, minute, second, time_of_day, is_night, 
delta_time_prev, delta_time_next

Features catégorielles encodées : toutes les colonnes node/opérations en OneHot

Label : normality

Vérifier les plages des features numériques

In [31]:
numeric_features = [
    'value_int', 'value_float',
    'hour', 'minute', 'second', 'time_of_day', 'is_night'
]

# afficher min, max, mean, std
data_tree[numeric_features].describe()


,value_int,value_float,hour,minute,second,time_of_day,is_night
count,357952.000000,357952.000000,357952.000000,357952.000000,357952.000000,357952.000000,357952.000000
mean,1.950222,19.965531,12.046693,29.453360,29.509937,12.537583,0.334098
std,2.494838,0.758331,6.772329,17.235144,17.283045,6.780850,0.471675
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,19.992900,6.000000,15.000000,14.000000,6.816667,0.000000
50%,1.000000,19.992900,13.000000,29.000000,30.000000,13.066667,0.000000
75%,1.000000,19.992900,18.000000,44.000000,44.000000,18.350000,1.000000
max,10.000000,38.571738,23.000000,59.000000,59.000000,23.983333,1.000000


In [32]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler


# Normalize time features
normalize_cols = [
    'hour', 'minute', 'second', 'time_of_day',
]

scaler_minmax = MinMaxScaler()
data_tree[normalize_cols] = scaler_minmax.fit_transform(data_tree[normalize_cols])

# Check the results
data_tree[normalize_cols].describe()


,hour,minute,second,time_of_day
count,357952.000000,357952.000000,357952.000000,357952.000000
mean,0.523769,0.499209,0.500168,0.522762
std,0.294449,0.292121,0.292933,0.282732
min,0.000000,0.000000,0.000000,0.000000
25%,0.260870,0.254237,0.237288,0.284225
50%,0.565217,0.491525,0.508475,0.544823
75%,0.782609,0.745763,0.745763,0.765115
max,1.000000,1.000000,1.000000,1.000000


data final :

value_numeric, delta_time_prev, delta_time_next	: Standardisées (mean=0, std=1)
hour, minute, second	: Normalisées [0,1]
value_bool, value_missing :	Binaires 0/1
Colonnes catégorielles  :	Encodées en nombres

In [33]:
display(data_tree.head(10))

,normality,value_is_missing,value_is_float,value_is_integer,value_is_timestamp_ms,value_is_boolean,value_is_java_object,value_is_string,value_int,value_float,...,accessedNodeType_/sensorService,accessedNodeType_/smartPhone,accessedNodeType_/thermostat,accessedNodeType_/washingService,accessedNodeType_unknown,operation_lockSubtree,operation_read,operation_registerService,operation_subscribe,operation_write
0,0,1,0,0,0,0,0,0,1.0,19.9929,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0,1,0,0,0,0,0,0,1.0,19.9929,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0,1,0,0,0,0,0,0,1.0,19.9929,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0,1,0,0,0,0,0,0,1.0,19.9929,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0,1,0,0,0,0,0,0,1.0,19.9929,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5,0,1,0,0,0,0,0,0,1.0,19.9929,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6,0,1,0,0,0,0,0,0,1.0,19.9929,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7,0,1,0,0,0,0,0,0,1.0,19.9929,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
8,0,1,0,0,0,0,0,0,1.0,19.9929,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0,1,0,0,0,0,0,0,1.0,19.9929,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [34]:
data_tree.to_csv("data/data_processed_for_tree-based_models.csv", index=False)

print("Preprocessed data exported successfully!")


Preprocessed data exported successfully!
